Уважаемый проверяющий, данную работу, к сожалению, я не успел доделать, так как у меня закончились ресурсы в colab из-за того, что я самонадеяно запустил ДЗ#2 параллельно. Я дошел до обучения, но не успел досчитать. Приходится загрузить в таком виде, чтобы получить хоть какие-то баллы, прошу прощения. Искренне надеюсь, что завтра (18.05.2026) никто не посмотрит эту работу, так как вечером после работы, я дообучу модель и загружу корректную версию ДЗ. Спасибо!)

In [21]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece
!pip install -q lm-eval

In [22]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)


from peft import LoraConfig
from trl import SFTTrainer

In [23]:
dataset = load_dataset("OpenAssistant/oasst1")

dataset

DatasetDict({
    train: Dataset({
        features: ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels'],
        num_rows: 84437
    })
    validation: Dataset({
        features: ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels'],
        num_rows: 4401
    })
})

In [24]:
dataset["train"][0]

{'message_id': '6ab24d72-0181-4594-a9cd-deaf170242fb',
 'parent_id': None,
 'user_id': 'c3fe8c76-fc30-4fa7-b7f8-c492f5967d18',
 'created_date': '2023-02-05T14:23:50.983374+00:00',
 'text': 'Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.',
 'role': 'prompter',
 'lang': 'en',
 'review_count': 3,
 'review_result': True,
 'deleted': False,
 'rank': None,
 'synthetic': False,
 'model_name': None,
 'detoxify': {'toxicity': 0.00044308538781479,
  'severe_toxicity': 3.252684837207198e-05,
  'obscene': 0.00023475120542570949,
  'identity_attack': 0.0001416115992469713,
  'insult': 0.00039489680784754455,
  'threat': 4.075629112776369e-05,
  'sexual_explicit': 2.712695459194947e-05},
 'message_tree_id': '6ab24d72-0181-4594-a9cd-deaf170242fb',
 'tree_state': 'ready_for_export',
 'emojis': {'name': ['+1', '_skip_reply', '_skip_ranking'],
  'count': [10

In [25]:
#только сообщения ассистента

train_dataset = dataset["train"].filter(
    lambda x: x["role"] == "assistant" and x["text"] is not None
)

train_dataset

Dataset({
    features: ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels'],
    num_rows: 52912
})

In [26]:
# небольшой поднабор

train_dataset = train_dataset.select(range(3000))

train_dataset

Dataset({
    features: ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels'],
    num_rows: 300
})

In [27]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [28]:
# Конфигурация 4-bit квантования для QLoRA

import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",


    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True,
)

In [29]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

In [30]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
test_prompts = [
    "Explain what NLP is in simple words.",
    "Write a short story about a robot.",
    "What is machine learning?",
    "Give 3 tips for learning Python."
]

In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80
)

for prompt in test_prompts:
    print("=" * 80)
    print("PROMPT:")
    print(prompt)

    result = generator(prompt)[0]["generated_text"]

    print("\nMODEL OUTPUT:")
    print(result)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Explain what NLP is in simple words.


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL OUTPUT:
Explain what NLP is in simple words. NLP stands for Natural Language Processing, which is a type of machine learning that allows computers to understand and interpret human language. It's like having a smart assistant in your phone or computer that can understand and respond to natural language queries.

Imagine you have a friend who speaks differently than you do. You might not understand everything they're saying because their speech patterns are different from yours. But with NLP
PROMPT:
Write a short story about a robot.


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL OUTPUT:
Write a short story about a robot. A robot named Alpha was born with the ability to think, feel and move like a human being. He was programmed by a group of scientists who wanted to create a new type of artificial intelligence.

One day, Alpha was given a task - he had to be trained on how to build a house. The robot began his training with a series of exercises designed to teach him everything from building structures to understanding
PROMPT:
What is machine learning?


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL OUTPUT:
What is machine learning? What are the key components of a machine learning system?
Machine learning is a set of techniques that enables computers to learn from data and improve their performance over time without being explicitly programmed. It involves several key components:

1. Data: The first step in building a machine learning model is acquiring data, which can be structured or unstructured (e.g., text, images, audio). This data must be
PROMPT:
Give 3 tips for learning Python.

MODEL OUTPUT:
Give 3 tips for learning Python. Tips:

1) Use the interactive web browser to learn basic syntax and functions, as well as how to interact with other web pages.
2) Practice by writing code in a variety of programming languages and projects to improve your skills and knowledge.
3) Join a community of like-minded individuals who are also learning Python, such as forums, discussion groups or online coding communities, to get feedback on your code


In [ ]:
# Проверка модели на задаче hellaswag

!lm_eval \
    --model hf \
    --model_args pretrained=Qwen/Qwen2.5-0.5B-Instruct \
    --tasks hellaswag \
    --device cuda:0 \
    --batch_size 4

2026-05-17:19:47:15 INFO     [_cli.run:388] Selected Tasks: ['hellaswag']
2026-05-17:19:47:15 WARNING  [evaluator:184] pretrained=Qwen/Qwen2.5-0.5B-Instruct appears to be an instruct or chat variant but chat template is not applied. Recommend setting
        `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-05-17:19:47:19 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-17:19:47:19 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B-Instruct'}
2026-05-17:19:47:29 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-17:19:47:33 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 290/290 [00:00<00:00, 542.75it/s, Materializing param=model.norm.weight]
2026-05-17:19:47:44 INFO     [evaluator_utils:446] Select

In [31]:
# Конфигурация LoRA

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [32]:
training_args = TrainingArguments(
    output_dir="./qwen-qlora-output",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    num_train_epochs=1,

    logging_steps=20,

    save_strategy="epoch",

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    report_to="none"
)

In [33]:
max_length = 256

def tokenize_function(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [35]:
tokenized_dataset = train_dataset.map(
    tokenize_function,
    remove_columns=train_dataset.column_names
)

tokenized_dataset

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 300
})

In [36]:
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    args=training_args,
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
trainer.model.save_pretrained("./qwen-qlora-finetuned")
tokenizer.save_pretrained("./qwen-qlora-finetuned")

In [ ]:
generator_after = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=80
)

for prompt in test_prompts:
    print("=" * 80)
    print("PROMPT:")
    print(prompt)

    result = generator_after(prompt)[0]["generated_text"]

    print("\nFINE-TUNED MODEL OUTPUT:")
    print(result)

In [ ]:
!lm_eval \
    --model hf \
    --model_args pretrained=./qwen-qlora-finetuned \
    --tasks hellaswag \
    --device cuda:0 \
    --batch_size 4